# Canonical attacks evaluated on APRIL-GAN

This notebook mirrors the complete existing Kaggle evaluation workflow for the official [APRIL-GAN](https://github.com/ByChelsea/VAND-APRIL-GAN) implementation. It clones pinned source, uses the released projection checkpoints from that repository, reads fixed attacks and evaluation IDs from the attached canonical Kaggle dataset, and packages numerical and qualitative results. Enable a GPU and Internet before running all cells.

The official zero-shot script uses OpenAI `ViT-L-14-336` (`ViT-L/14@336px`) with 518-pixel inputs and feature layers 6/12/18/24. The zero-shot mapping is deliberate: MVTec uses `visa_pretrained.pth`, while VisA uses `mvtec_pretrained.pth`. APRIL-GAN anomaly maps are evaluated without Gaussian smoothing (`anomaly_map_sigma=0`).

Blackbox protocol constraint: the shared loader resizes every clean image, adversarial image, and ground-truth mask directly to the same 518x518 square coordinate system. This preserves pixel alignment with the canonical square attack tensors, but differs from APRIL-GAN's aspect-ratio-preserving resize plus center crop on non-square source images.


In [ ]:
import hashlib
import shutil
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE REPOSITORIES AND INSTALL DEPENDENCIES =====')
WORKING = Path('/kaggle/working')
EXPERIMENT_ROOT = WORKING / 'adversarial-robustness'
APRILGAN_ROOT = WORKING / 'VAND-APRIL-GAN'
EXPERIMENT_REPO_URL = 'https://github.com/Parsagh05/adversarial-robustness.git'
APRILGAN_REPO_URL = 'https://github.com/ByChelsea/VAND-APRIL-GAN.git'
APRILGAN_COMMIT = 'f13b8a634e04f9fde8fa03db125b25af5695d8e1'

def clone_or_update(url, destination, commit=None):
    if destination.exists():
        subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    else:
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    if commit:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    else:
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)

clone_or_update(EXPERIMENT_REPO_URL, EXPERIMENT_ROOT)
clone_or_update(APRILGAN_REPO_URL, APRILGAN_ROOT, APRILGAN_COMMIT)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'requirements.txt')
], check=True)
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))

# APRIL-GAN's vendored OpenCLIP loader uses this OpenAI ViT-L/14@336px checkpoint.
BASE_MODEL_NAME = 'ViT-L-14-336px.pt'
BASE_MODEL_SHA256 = '3035c92b350959924f9f00213499208652fc7ea050643e8b385c2dac08641f02'
BASE_MODEL_URL = (
    'https://openaipublic.azureedge.net/clip/models/'
    + BASE_MODEL_SHA256 + '/' + BASE_MODEL_NAME
)
APRILGAN_CLIP_CACHE = WORKING / 'aprilgan_clip_cache'
APRILGAN_CLIP_CACHE.mkdir(parents=True, exist_ok=True)
BASE_MODEL_PATH = APRILGAN_CLIP_CACHE / BASE_MODEL_NAME

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

attached_base = next(
    (path for path in Path('/kaggle/input').rglob(BASE_MODEL_NAME) if path.is_file()),
    None,
)
if not BASE_MODEL_PATH.is_file() or sha256(BASE_MODEL_PATH) != BASE_MODEL_SHA256:
    if attached_base is not None:
        if sha256(attached_base) != BASE_MODEL_SHA256:
            raise RuntimeError(f'Attached base model has wrong SHA256: {attached_base}')
        shutil.copy2(attached_base, BASE_MODEL_PATH)
    else:
        import torch
        torch.hub.download_url_to_file(
            BASE_MODEL_URL, str(BASE_MODEL_PATH),
            hash_prefix=BASE_MODEL_SHA256, progress=True,
        )
if sha256(BASE_MODEL_PATH) != BASE_MODEL_SHA256:
    raise RuntimeError(f'Base-model checksum mismatch: {BASE_MODEL_PATH}')
print('Experiment code:', EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline')
print('Official APRIL-GAN:', APRILGAN_ROOT)
print('Verified base model:', BASE_MODEL_PATH)


In [ ]:
from blackbox_evaluation_pipeline.universal_eval.artifacts import load_manifest

print('===== STEP 2: RESOLVE THE ATTACHED CANONICAL ATTACK DATASET =====')
ATTACK_SCOPES = ('per_dataset',)  # per_dataset, per_category, per_image
ATTACK_SOURCE_DATASETS = None
ATTACK_TARGET_DATASETS = ('mvtec', 'visa')
ATTACK_CATEGORIES = None
ATTACK_DIRECTIONS = None
ATTACK_LOSS_MODES = None

def valid_artifact_root(path):
    return path.is_dir() and all(
        (path / f'canonical_clip_{scope}' / 'attack_manifest.csv').is_file()
        for scope in ATTACK_SCOPES
    )

candidates = [
    Path('/kaggle/input/datasets/alirezasalehy/adversarial-attacks-vlm-survey'),
    Path('/kaggle/input/adversarial-attacks-vlm-survey'),
]
if Path('/kaggle/input').is_dir():
    candidates.extend(path.parent for path in Path('/kaggle/input').rglob('canonical_clip_per_dataset'))
ARTIFACTS_ROOT = next((path for path in candidates if valid_artifact_root(path)), None)
if ARTIFACTS_ROOT is None:
    raise FileNotFoundError('Attach alirezasalehy/adversarial-attacks-vlm-survey.')
artifacts = load_manifest(
    ARTIFACTS_ROOT, scopes=ATTACK_SCOPES, sources=ATTACK_SOURCE_DATASETS,
    targets=ATTACK_TARGET_DATASETS, categories=ATTACK_CATEGORIES,
    directions=ATTACK_DIRECTIONS, loss_modes=ATTACK_LOSS_MODES,
)
print('Canonical root:', ARTIFACTS_ROOT)
print('Available conditions:', len(artifacts))
for source, target in sorted({(a.record['source_dataset'], a.record['target_dataset']) for a in artifacts}):
    count = sum(a.record['source_dataset'] == source and a.record['target_dataset'] == target for a in artifacts)
    print(f'  {source} -> {target}: {count}')


In [ ]:
import torch

print('===== STEP 3: RESOLVE DATASETS AND ZERO-SHOT APRIL-GAN CHECKPOINTS =====')

def first_existing_directory(paths, label):
    for path in paths:
        if path.is_dir():
            return path
    raise FileNotFoundError(f'{label} was not found. Checked: {paths}')

MVTEC_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection'),
    Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'),
], 'MVTec AD')
VISA_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922'),
    Path('/kaggle/input/visa-ad/VisA_20220922'),
], 'VisA')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')

CHECKPOINT_ROOT = APRILGAN_ROOT / 'exps' / 'pretrained'
TRAIN_MVTEC_CHECKPOINT = CHECKPOINT_ROOT / 'mvtec_pretrained.pth'
TRAIN_VISA_CHECKPOINT = CHECKPOINT_ROOT / 'visa_pretrained.pth'
for path in (TRAIN_MVTEC_CHECKPOINT, TRAIN_VISA_CHECKPOINT):
    if not path.is_file():
        raise FileNotFoundError(f'Released APRIL-GAN checkpoint not found: {path}')

MODEL_KWARGS_BY_TARGET = {
    'mvtec': {
        'repository_root': str(APRILGAN_ROOT),
        'checkpoint_path': str(TRAIN_VISA_CHECKPOINT),
        'clip_cache_dir': str(APRILGAN_CLIP_CACHE),
    },
    'visa': {
        'repository_root': str(APRILGAN_ROOT),
        'checkpoint_path': str(TRAIN_MVTEC_CHECKPOINT),
        'clip_cache_dir': str(APRILGAN_CLIP_CACHE),
    },
}
print('MVTec:', MVTEC_ROOT)
print('VisA:', VISA_ROOT)
print('MVTec target <- VisA weights:', TRAIN_VISA_CHECKPOINT)
print('VisA target <- MVTec weights:', TRAIN_MVTEC_CHECKPOINT)


In [ ]:
import json

from blackbox_evaluation_pipeline import EvaluationConfig, run_evaluation

print('===== STEP 4: RUN FIXED-ID CLEAN/ADVERSARIAL EVALUATION =====')
FULL_RUN = True
OUTPUT_ROOT = WORKING / ('kaggle_new_aprilgan_full' if FULL_RUN else 'kaggle_new_aprilgan_check')
SAMPLES_ROOT = WORKING / ('kaggle_new_aprilgan_samples_full' if FULL_RUN else 'kaggle_new_aprilgan_samples_check')

def normalized_name(value):
    return ''.join(character for character in value.lower() if character.isalnum())

def find_f1_threshold(dataset, model_name):
    committed = (
        EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'thresholds'
        / normalized_name(model_name) / dataset / 'category_thresholds.json'
    )
    candidates = [committed] if committed.is_file() else list(Path('/kaggle/input').rglob('category_thresholds.json'))
    valid = []
    for path in candidates:
        try:
            payload = json.loads(path.read_text(encoding='utf-8'))
        except (OSError, ValueError):
            continue
        if (
            payload.get('dataset') == dataset
            and normalized_name(str(payload.get('target_model', ''))) == normalized_name(model_name)
            and payload.get('threshold_mode') == 'clean_f1_optimal'
        ):
            valid.append(path)
    if len(valid) != 1:
        raise RuntimeError(
            f'Expected one clean F1-optimal {model_name}/{dataset} artifact, found {valid}. '
            'Run kaggle_new_thresholds.ipynb and attach or commit its verified output.'
        )
    return str(valid[0])

THRESHOLDS_BY_TARGET = {
    dataset: find_f1_threshold(dataset, 'aprilgan')
    for dataset in ATTACK_TARGET_DATASETS
}
print('Frozen clean F1-optimal thresholds:', THRESHOLDS_BY_TARGET)
config = EvaluationConfig(
    artifacts_root=str(ARTIFACTS_ROOT), mvtec_root=str(MVTEC_ROOT),
    visa_root=str(VISA_ROOT), output_root=str(OUTPUT_ROOT),
    model_name='aprilgan', model_kwargs_by_target=MODEL_KWARGS_BY_TARGET,
    thresholds_by_target=THRESHOLDS_BY_TARGET, device='cuda', batch_size=2,
    metric_size=518, anomaly_map_sigma=0.0, aupro_fpr_limit=0.30,
    aupro_max_thresholds=200, verify_checksums=True, save_predictions=True,
    prediction_map_size=37, save_qualitative_samples=True,
    qualitative_output_root=str(SAMPLES_ROOT), attack_scopes=ATTACK_SCOPES,
    source_datasets=ATTACK_SOURCE_DATASETS, target_datasets=ATTACK_TARGET_DATASETS,
    attack_categories=ATTACK_CATEGORIES, attack_directions=ATTACK_DIRECTIONS,
    attack_loss_modes=ATTACK_LOSS_MODES, max_conditions=None if FULL_RUN else 1,
    run_notes=(
        'Attached canonical CSV attack bundles; fixed evaluation IDs; official '
        'APRIL-GAN OpenAI ViT-L/14@336px and opposite-dataset zero-shot weights; '
        'official unsmoothed anomaly maps; shared direct 518x518 square resize for '
        'clean/adversarial image and mask alignment.'
    ),
)
SUMMARY_PATH = run_evaluation(config)
print('Finished:', SUMMARY_PATH)


In [ ]:
import csv

print('===== STEP 5: PREVIEW SUMMARY =====')
with SUMMARY_PATH.open(newline='', encoding='utf-8') as handle:
    summary_rows = list(csv.DictReader(handle))
columns = [
    'source_dataset', 'target_dataset', 'scope', 'category', 'direction', 'loss_mode',
    'clean_i_auroc', 'adversarial_i_auroc', 'delta_i_auroc',
    'clean_p_auroc', 'adversarial_p_auroc', 'delta_p_auroc',
    'clean_aupro', 'adversarial_aupro', 'delta_aupro',
    'clean_accuracy', 'adversarial_accuracy',
    'clean_fpr', 'adversarial_fpr', 'clean_fnr', 'adversarial_fnr',
    'attack_flip_rate', 'targeted_attack_success_rate',
]
for row in summary_rows:
    print({column: row[column] for column in columns})


In [ ]:
print('===== STEP 6: PACKAGE OUTPUTS =====')
results_archive = shutil.make_archive(
    str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name
)
samples_archive = shutil.make_archive(
    str(SAMPLES_ROOT), 'zip', root_dir=SAMPLES_ROOT.parent, base_dir=SAMPLES_ROOT.name
)
print('Packaged numerical results:', results_archive)
print('Packaged qualitative samples:', samples_archive)
